# GET CONTACT INFO

In [7]:
# -----######-----###### CORE FUNCTION -----######-----######
import sqlite3
import pandas as pd
import re

def _contacts_1803_i3_GET_df_contacts_UNIFIED(contacts_db_path):

    conn = sqlite3.connect(contacts_db_path)

    # pull unified contacts
    query = """
    SELECT 
        r.Z_PK as contact_id,
        r.ZFIRSTNAME,
        r.ZLASTNAME,
        r.ZORGANIZATION,
        p.ZFULLNUMBER as phone,
        e.ZADDRESS as email,
        n.ZNOTE
    FROM ZABCDRECORD r
    LEFT JOIN ZABCDPHONENUMBER p ON r.Z_PK = p.ZOWNER
    LEFT JOIN ZABCDEMAILADDRESS e ON r.Z_PK = e.ZOWNER
    LEFT JOIN ZABCDNOTE n ON r.Z_PK = n.ZOWNER
    """

    df = pd.read_sql_query(query, conn)
    conn.close()

    # clean name
    df["name"] = (
        df["ZFIRSTNAME"].fillna("") + " " + df["ZLASTNAME"].fillna("")
    ).str.strip()

    # clean phone
    df["phone_clean"] = df["phone"].apply(
        lambda x: re.sub(r"\D", "", str(x)) if pd.notnull(x) else None
    )

    df = df[[
        "contact_id",
        "name",
        "ZORGANIZATION",
        "phone_clean",
        "email",
        "ZNOTE"
    ]]

    df.columns = [
        "contact_id",
        "name",
        "company",
        "phone_clean",
        "email",
        "notes"
    ]

    # remove empty rows
    df = df[(df["phone_clean"].notna()) | (df["email"].notna())]

    return df

In [8]:
# !#!#!#!#! RUNNING STATEMENTS !#!#!#!#!

contacts_db_path = "/Users/yerik/Library/Application Support/AddressBook/AddressBook-v22.abcddb"

df_contacts_clean = _contacts_1803_i3_GET_df_contacts_UNIFIED(contacts_db_path)

df_contacts_clean.head(20)

DatabaseError: Execution failed on sql '
    SELECT 
        r.Z_PK as contact_id,
        r.ZFIRSTNAME,
        r.ZLASTNAME,
        r.ZORGANIZATION,
        p.ZFULLNUMBER as phone,
        e.ZADDRESS as email,
        n.ZNOTE
    FROM ZABCDRECORD r
    LEFT JOIN ZABCDPHONENUMBER p ON r.Z_PK = p.ZOWNER
    LEFT JOIN ZABCDEMAILADDRESS e ON r.Z_PK = e.ZOWNER
    LEFT JOIN ZABCDNOTE n ON r.Z_PK = n.ZOWNER
    ': no such column: n.ZNOTE

# GET DF with contacts 

In [15]:
# -----######-----###### CORE FUNCTION -----######-----######
import sqlite3
import pandas as pd
import re
from tqdm import tqdm
from datetime import datetime, timedelta

def _msg_1803_i6_GET_df_contacts_FIXED(msg_db_path, contacts_db_path):

    def clean_phone(x):
        return re.sub(r"\D", "", x) if x else None

    def apple_to_datetime(apple_time):
        if apple_time:
            return datetime(2001, 1, 1) + timedelta(seconds=apple_time)
        return None

    # --- LOAD CONTACTS DB ---
    conn_ct = sqlite3.connect(contacts_db_path)
    cur_ct = conn_ct.cursor()

    contact_lookup = {}

    rows = cur_ct.execute("""
        SELECT 
            c.Z_PK,
            c.ZFIRSTNAME,
            c.ZLASTNAME,
            c.ZORGANIZATION,
            c.ZNOTE,
            c.ZBIRTHDAY,
            p.ZFULLNUMBER,
            e.ZADDRESS
        FROM ZABCDRECORD c
        LEFT JOIN ZABCDPHONENUMBER p ON c.Z_PK = p.ZOWNER
        LEFT JOIN ZABCDEMAILADDRESS e ON c.Z_PK = e.ZOWNER
    """).fetchall()

    for pk, fname, lname, org, note, bday, phone, email in rows:

        name = f"{fname or ''} {lname or ''}".strip()

        if phone:
            key = clean_phone(phone)
            contact_lookup[key] = {
                "name": name,
                "company": org,
                "notes": note,
                "birthday": apple_to_datetime(bday)
            }

        if email:
            contact_lookup[email] = {
                "name": name,
                "company": org,
                "notes": note,
                "birthday": apple_to_datetime(bday)
            }

    conn_ct.close()

    # --- LOAD MESSAGE DB ---
    conn_msg = sqlite3.connect(msg_db_path)
    cur_msg = conn_msg.cursor()

    handles = cur_msg.execute("""
        SELECT ROWID, id
        FROM handle
    """).fetchall()

    data = []

    print("Processing contacts...")
    for rowid, contact_id in tqdm(handles):

        count = cur_msg.execute(f"""
            SELECT COUNT(*)
            FROM message
            WHERE handle_id = {rowid}
        """).fetchone()[0]

        is_email = "@" in contact_id
        phone_clean = clean_phone(contact_id) if not is_email else None

        info = None

        if phone_clean and phone_clean in contact_lookup:
            info = contact_lookup[phone_clean]

        elif contact_id in contact_lookup:
            info = contact_lookup[contact_id]

        data.append({
            "handle_id": rowid,
            "contact_raw": contact_id,
            "contact_type": "email" if is_email else "phone",
            "phone_clean": phone_clean,
            "msg_count": count,
            "name": info["name"] if info else None,
            "company": info["company"] if info else None,
            "notes": info["notes"] if info else None,
            "birthday": info["birthday"] if info else None
        })

    conn_msg.close()

    df = pd.DataFrame(data)
    df = df.sort_values("msg_count", ascending=False).reset_index(drop=True)

    return df

In [16]:
# !#!#!#!#! RUNNING STATEMENTS !#!#!#!#!

msg_db_path = "/Users/yerik/Library/Messages/chat.db"
contacts_db_path = "/Users/yerik/Library/Application Support/AddressBook/AddressBook-v22.abcddb"

df_contacts = _msg_1803_i6_GET_df_contacts_FIXED(msg_db_path, contacts_db_path)

df_contacts.head(20)

Processing contacts...


100%|██████████████████████████████████████████████████████████████████| 1765/1765 [00:00<00:00, 59682.89it/s]


,handle_id,contact_raw,contact_type,phone_clean,msg_count,name,company,notes,birthday
0,12,+12489806843,phone,12489806843,10957,None,None,None,None
1,41,+12487590280,phone,12487590280,1930,None,None,None,None
2,45,+12484048679,phone,12484048679,1851,None,None,None,None
3,65,+18106270297,phone,18106270297,1717,None,None,None,None
4,83,+12484085144,phone,12484085144,1677,None,None,None,None
5,24,+12489493330,phone,12489493330,1217,None,None,None,None
6,75,+12487908319,phone,12487908319,1140,None,None,None,None
7,119,+12483906173,phone,12483906173,1138,None,None,None,None
8,882,+13129520107,phone,13129520107,989,None,None,None,None
9,276,+13135057455,phone,13135057455,862,None,None,None,None


# pull messages from someone 

In [1]:
# -----######-----###### CORE FUNCTION -----######-----######
import sqlite3
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from datetime import datetime

def _msg_1803_i1_GET_pdf_from_contact(db_path, contact, output_pdf):

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    query = f"""
    SELECT 
        datetime(message.date/1000000000 + 978307200, 'unixepoch', 'localtime') as msg_date,
        message.text,
        message.is_from_me
    FROM message
    JOIN handle ON message.handle_id = handle.ROWID
    WHERE handle.id LIKE '%{contact}%'
    AND message.text IS NOT NULL
    ORDER BY message.date ASC
    """

    rows = cursor.execute(query).fetchall()
    conn.close()

    styles = getSampleStyleSheet()
    doc = SimpleDocTemplate(output_pdf)

    content = []

    for date, text, is_me in rows:

        sender = "Me" if is_me else contact
        line = f"<b>{sender}</b> [{date}]: {text}"

        content.append(Paragraph(line, styles["Normal"]))
        content.append(Spacer(1, 8))

    doc.build(content)

    print(f"✅ PDF created: {output_pdf}")

In [2]:
# !#!#!#!#! RUNNING STATEMENTS !#!#!#!#!

db_path = "/Users/yerik/Library/Messages/chat.db"
contact = "12489806843	"   # <- put spouse name or number
output_pdf = "_messages_export/NUCHO.pdf"

_msg_1803_i1_GET_pdf_from_contact(db_path, contact, output_pdf)

✅ PDF created: _messages_export/NUCHO.pdf


In [3]:
# search directly in Contacts DB
import sqlite3

conn = sqlite3.connect("/Users/yerik/Library/Application Support/AddressBook/AddressBook-v22.abcddb")
cur = conn.cursor()

rows = cur.execute("""
SELECT ZFIRSTNAME, ZLASTNAME, ZFULLNUMBER
FROM ZABCDRECORD c
JOIN ZABCDPHONENUMBER p ON c.Z_PK = p.ZOWNER
WHERE ZFULLNUMBER LIKE '%2489806843%'
""").fetchall()

for r in rows:
    print(r)

conn.close()

In [6]:
import sqlite3

conn = sqlite3.connect("/Users/yerik/Library/Application Support/AddressBook/AddressBook-v22.abcddb")
cur = conn.cursor()

tables = cur.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()

for t in tables:
    print(t)

conn.close()

('ZABCDADDRESSINGGRAMMAR',)
('ZABCDALERTTONE',)
('ZABCDCALENDARURI',)
('ZABCDCONTACTDATE',)
('ZABCDCONTACTINDEX',)
('ZABCDCUSTOMPROPERTY',)
('ZABCDCUSTOMPROPERTYVALUE',)
('ZABCDDATECOMPONENTS',)
('ZABCDDELETEDRECORDLOG',)
('ZABCDDISTRIBUTIONLISTCONFIG',)
('ZABCDEMAILADDRESS',)
('ZABCDLIKENESS',)
('ZABCDMESSAGINGADDRESS',)
('ZABCDNOTE',)
('ZABCDPHONENUMBER',)
('ZABCDPOSTALADDRESS',)
('ZABCDRECORD',)
('Z_18PARENTGROUPS',)
('Z_22PARENTGROUPS',)
('ZABCDRELATEDNAME',)
('ZABCDREMOTELOCATION',)
('ZABCDSERVICE',)
('ZABCDSOCIALPROFILE',)
('ZABCDUNKNOWNPROPERTY',)
('ZABCDURLADDRESS',)
('ZCNCDCHANGEHISTORYCLIENT',)
('ZCNCDPROVIDERMETADATA',)
('ZCNCDUNIFIEDCONTACTINFO',)
('Z_PRIMARYKEY',)
('Z_METADATA',)
('Z_MODELCACHE',)
('ACHANGE',)
('ATRANSACTION',)
('ATRANSACTIONSTRING',)
